# Fitur ResNet50 + Isolation Forest
        **vision › anomaly** · notebook `resnet_features_isolation_forest`

        **Kapan dipakai:** Pendekatan praktis untuk anomali visual: fitur pretrained ditambah detektor tabular klasik.

        **Jebakan umum:** Bergantung pada kecocokan fitur ImageNet dengan domain; anomali halus (retakan kecil) butuh fitur lokal/patch.

> ⚠️ Notebook ini berat (butuh torch / bobot / GPU). Lihat sel pertama.

**Data demo:** CIFAR-10 subset (4000 latih / 1000 uji, 10 kelas, 32x32).

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
DATA_ROOT = ROOT / "data_latihan/vision/classification/cifar10_small"
NORMAL = 'automobile'      # kelas yang dianggap NORMAL (hanya ini yang boleh dilihat saat melatih)
SEED, EPOCHS = 42, 25
MODEL_KEY = "resnet_features_isolation_forest"
OUT = out_dir("vision", "anomaly", MODEL_KEY); set_seed(SEED)

## 2. Data

In [ ]:
import torch, torch.nn as nn
from superdb.vision import load_images, device, show_grid
tr = pd.read_csv(DATA_ROOT / "train.csv"); te = pd.read_csv(DATA_ROOT / "test.csv").merge(pd.read_csv(DATA_ROOT / "test_labels.csv"), on="id")
X, Xte = load_images(tr["path"], DATA_ROOT), load_images(te["path"], DATA_ROOT); y, yte = tr["label"].to_numpy(), te["label"].to_numpy()
print(X.shape, Xte.shape, "| device:", device())

## 3. Skenario
Latih HANYA pada kelas `NORMAL`. Uji: gambar normal (label 0) vs semua kelas lain (anomali, label 1).

In [ ]:
Xn = X[y == NORMAL]; is_anom = (yte != NORMAL).astype(int)
print(f"latih: {len(Xn)} gambar normal | uji: {int((is_anom == 0).sum())} normal + {int(is_anom.sum())} anomali")
from sklearn.metrics import roc_auc_score, average_precision_score

## 4. Fitur ResNet50 beku + Isolation Forest (dilatih hanya pada fitur gambar normal)

In [ ]:
from superdb.vision import extract_features
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
Fn, Fte = extract_features("resnet50", Xn, 96, tag="normal"), extract_features("resnet50", Xte, 96)
det = make_pipeline(StandardScaler(), PCA(n_components=50, random_state=SEED), IsolationForest(n_estimators=300, random_state=SEED, n_jobs=-1)).fit(Fn)
score = -det.score_samples(Fte)

## 5. Evaluasi
PR-AUC dibanding garis dasar (proporsi anomali).

In [ ]:
metrics = {"roc_auc": float(roc_auc_score(is_anom, score)), "pr_auc": float(average_precision_score(is_anom, score)), "baseline_pr_auc": float(is_anom.mean())}; display(show_table(metrics))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2)); ax[0].hist(score[is_anom == 0], bins=30, alpha=.6, density=True, label="normal"); ax[0].hist(score[is_anom == 1], bins=30, alpha=.6, density=True, label="anomali"); ax[0].legend(); ax[0].set_title("distribusi skor")
top = np.argsort(-score)[:10]; ax[1].imshow(np.hstack([Xte[i] for i in top[:8]])); ax[1].axis("off"); ax[1].set_title("8 gambar paling 'aneh'")
plt.tight_layout(); plt.savefig(OUT / "anomaly.png", dpi=130); plt.show()
save_json({"metrics": metrics}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Anomali visual nyata (cacat produk): fitur patch-level (PatchCore) jauh lebih peka daripada fitur seluruh gambar.
- Evaluasi anomali selalu dengan PR-AUC; ROC-AUC bisa menyesatkan bila anomali langka.